In [19]:
import numpy as np
import pandas as pd
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
import pytz
import yfinance as yf
import pyodbc
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time
import logging
import pandas as pd
from truedata import TD_hist
import requests

In [20]:
def fetch_truedata_history(
    ticker_list: list,
    duration: str = '1 Y',
    bar_size: str = 'EOD',
    sleep_time: float = 0.1
) -> tuple[pd.DataFrame, list]:
    """
    Fetches historical data from TrueData for a list of tickers.

    Parameters
    ----------
    username : str
        TrueData username.
    password : str
        TrueData password.
    ticker_list : list
        List of ticker symbols to fetch data for.
    duration : str, optional
        Duration of data (e.g., '1 Y', '25 Y', etc.). Default is '1 Y'.
    bar_size : str, optional
        Bar size for data ('EOD', 'WEEK', etc.). Default is 'EOD'.
    sleep_time : float, optional
        Delay between API calls to avoid throttling. Default is 0.2 seconds.

    Returns
    -------
    final_df : pd.DataFrame
        Combined DataFrame of all tickers' historical data.
    error_list : list
        List of tickers that failed to fetch.
    """
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    username = 'tdwsf695'
    password = 'ocean@695'
    # Initialize connection
    td_hist = TD_hist(username, password)
    df_list = []
    error_list = []
    for ticker in ticker_list:
        try:
            df = td_hist.get_historic_data([ticker], duration=duration, bar_size=bar_size)

            df['Ticker'] = ticker
            # Check column names and rename accordingly
            rename_dict = {}
            if 'timestamp' in df.columns:
                rename_dict['timestamp'] = 'Date'
            elif 'datetime' in df.columns:
                rename_dict['datetime'] = 'Date'
            elif 'date' in df.columns:
                rename_dict['date'] = 'Date'
            rename_dict.update({
                'high': 'High',
                'low': 'Low',
                'close': 'Close',
                'open': 'Open'
            })
            df = df.rename(columns=rename_dict)

            df_list.append(df)
            logging.info(f"Fetched data for {ticker} ({len(df)} rows).")
            time.sleep(sleep_time)

        except Exception as e:
            logging.error(f"Failed to fetch data for {ticker}: {e}")
            error_list.append(ticker)

    final_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
    return final_df, error_list

In [21]:
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta

def process_portfolio(nav_df, ticker_data, initial_value=75, output_file=None):
    """
    Process portfolio allocation and returns final dataframe with portfolio performance.

    Parameters
    ----------
    nav_df : pd.DataFrame
        Dataframe with at least ['Year-Month', 'Ticker'] columns.
    get_individual_stock_data : function
        Function to fetch OHLC data. Must accept (tickers, start_date, end_date) and return DataFrame with ['Date','Ticker','Close'].
    initial_value : float
        Initial portfolio allocation value (default=75).
    debt_ticker : str
        Ticker used as debt/alternative asset (default 'MOGSEC.NS').
    output_file : str or None
        If provided, saves the final dataframe to Excel.

    Returns
    -------
    pd.DataFrame
        Final dataframe with portfolio values.
    """
    df_lis = []
    last_month_value = {}
    year_months = nav_df['Year-Month'].unique()
    for i, year_month in enumerate(year_months):
        print(f"\nProcessing: {year_month}")
        print("Last Month Value:", last_month_value)

        tickers = nav_df[nav_df['Year-Month'] == year_month]['Ticker'].unique()
        year_month_date = pd.to_datetime(f"{year_month}-01")


        prev_month_start = (year_month_date - relativedelta(months=2)).strftime('%Y-%m-%d')
        curr_month_start = year_month_date.strftime('%Y-%m-%d')
        curr_month_end = (year_month_date + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

        # --- Fetch stock data ---
        # stock_data = get_individual_stock_data(tickers, prev_month_start, curr_month_end)
        stock_data = (
            ticker_data[(ticker_data['Date'] >= prev_month_start)
            & (ticker_data['Date'] <= curr_month_end) 
            & (ticker_data['Ticker'].isin(tickers))])
        
        # % change
        stock_data['%change'] = stock_data.groupby('Ticker')['Close'].pct_change()
        # Filter current month
        stock_data_flt = stock_data[
            (stock_data['Date'] >= curr_month_start) & (stock_data['Date'] <= curr_month_end)
        ].copy()

        print(stock_data_flt)

        # --- Portfolio allocation logic ---
        if len(last_month_value) == 0:
            # First month → allocate initial portfolio equally
            allocation_per_stock = initial_value / len(tickers)
            stock_allocations = {t: allocation_per_stock for t in tickers}
        else:
            # Continue portfolio
            stock_allocations = {t: last_month_value[t] for t in tickers if t in last_month_value}

            # Pool value of dropped stocks
            dropped_stocks = [t for t in last_month_value if t not in tickers]
            dropped_value = sum(last_month_value[t] for t in dropped_stocks)

            # New stocks → share the dropped value equally
            new_stocks = [t for t in tickers if t not in last_month_value]
            if new_stocks:
                allocation_per_stock = dropped_value / len(new_stocks)
                for t in new_stocks:
                    stock_allocations[t] = allocation_per_stock

        # Apply allocations into dataframe
        for tkr, init_value in stock_allocations.items():
            tkr_idx = stock_data_flt[stock_data_flt['Ticker'] == tkr].index
            stock_data_flt.loc[tkr_idx, 'Buy_Hold_Value'] = init_value * (
                (1 + stock_data_flt.loc[tkr_idx, '%change'].fillna(0)).cumprod()
            )

        # --- Update last month values ---
        last_month_value = (
            stock_data_flt.groupby('Ticker')['Buy_Hold_Value'].last().to_dict()
        )

        # --- Track total portfolio value ---
        stock_data_flt['Total_Portfolio_Value'] = (
            stock_data_flt.groupby('Date')['Buy_Hold_Value'].transform('sum')
        )

        df_lis.append(stock_data_flt)

    
    final_df = pd.concat(df_lis).reset_index(drop=True)

    if output_file:
        final_df.to_excel(output_file, index=False)

    return final_df


In [22]:
import os
import pandas as pd

def prepare_and_process_portfolio(input_file, start_date, end_date, output_folder,
                                  process_portfolio,
                                  equity_allocation=75, gold_allocation=25):
    """
    Prepare portfolio dataframe with momentum stocks + GOLDBEES and process performance.

    Parameters
    ----------
    input_file : str
        Path to momentum Excel file (with End_Date, Ticker columns).
    start_date : str (YYYY-MM-DD)
        Start date for filtering.
    end_date : str (YYYY-MM-DD)
        End date for filtering.
    output_folder : str
        Folder to save output file.
    get_individual_stock_data : function
        Function to fetch stock NAV/price data.
    process_pocrtfolio : function
        Function to process equity portion of portfolio.
    process_gold : function
        Function to process gold portion of portfolio.
    equity_allocation : int, optional
        Initial allocation to equities (default=75000).
    gold_allocation : int, optional
        Initial allocation to gold (default=25000).

    Returns
    -------
    final_df : pd.DataFrame
        Combined portfolio dataframe.
    """

    # Load and clean
    nav_df = pd.read_excel(input_file).rename(columns={'End_Date': 'Date'})
    nav_df['Date'] = pd.to_datetime(nav_df['Date'])
    nav_df = (
        nav_df[(nav_df['Date'] >= start_date) & (nav_df['Date'] <= end_date)]
        .reset_index(drop=True)[['Date', 'Ticker']]
    )
    nav_df['Year-Month'] = nav_df['Date'].dt.to_period('M').astype(str)
    stocks = pd.read_excel(input_file)

    # Add GOLDBEES for each unique date
    goldbees_df = pd.DataFrame({
        'Date': nav_df['Date'].unique(),
        'Ticker': 'GOLDBEES'
    })
    goldbees_df['Year-Month'] = pd.to_datetime(goldbees_df['Date']).dt.to_period('M').astype(str)
    # print(goldbees_df)

    # Combine
    concat_df = (
        pd.concat([nav_df, goldbees_df], ignore_index=True)
          .sort_values(['Date', 'Ticker'])
          .reset_index(drop=True)
    )

    # symbol_list = stocks['Ticker'].unique()
    # ticker_data = fetch_truedata_history(
    #     ticker_list = symbol_list,
    #     duration = '5 Y',
    #     bar_size = 'EOD',
    #     sleep_time= 0.1
    # )[0]
    # final_df = process_portfolio(concat_df, ticker_data, equity_allocation)

    
    # Split
    ticker_df = concat_df.query("Ticker != 'GOLDBEES'")
    symbol_list = ticker_df['Ticker'].unique()
    ticker_data_other_stocks = fetch_truedata_history(
        ticker_list = symbol_list,
        duration = '10 Y',
        bar_size = 'EOD',
        sleep_time= 0.1
    )[0]

    
    gold_df = concat_df.query("Ticker == 'GOLDBEES'")
    symbol_list = gold_df['Ticker'].unique()
    ticker_data_gold = fetch_truedata_history(
        ticker_list = symbol_list,
        duration = '10 Y',
        bar_size = 'EOD',
        sleep_time= 0.1
    )[0]
    # print(gold_df)

    # Process
    final_df_other_stocks = process_portfolio(ticker_df, ticker_data_other_stocks, equity_allocation)
    # final_df_gold = process_gold(gold_df, get_individual_stock_data, gold_allocation)
    final_df_gold = process_portfolio(gold_df, ticker_data_gold, gold_allocation)


    # Merge results
    final_df = (
        pd.concat([final_df_other_stocks, final_df_gold], ignore_index=True)
          .sort_values(['Date', 'Ticker'])
          .reset_index(drop=True)
    )


    # --- ensure output folder exists ---
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # --- extract middle folder name from input path ---
    middle_folder = os.path.basename(os.path.dirname(input_file))
    # e.g. for path ".../nifty500_21April2025_results/master_momentum_summary.xlsx"
    # middle_folder = "nifty500_21April2025_results"

    # --- create output filename using middle folder ---
    output_file = os.path.join(output_folder, f"{middle_folder}_gold_buy&hold_returns.xlsx")

    # --- save output ---
    # final_df.to_excel(output_file, index=False)
    print(f"✅ Final output saved to: {output_file}")

    return final_df


In [23]:
#NSE500

In [24]:
final_df = prepare_and_process_portfolio(
    input_file="Stocks/Nifty_500_2025_Apr_20_stocks_results/master_momentum_summary.xlsx",
    start_date="2023-04-01",
    end_date=date.today().strftime('%Y-%m-%d'),
    output_folder="Trials",
    process_portfolio=process_portfolio
)

import plotly.express as px

# ✅ Group by Date and calculate total portfolio value
portfolio_summary = (
    final_df.groupby("Date", as_index=False)["Buy_Hold_Value"].sum()
)

# ✅ Plot with Plotly
fig = px.line(
    portfolio_summary,
    x="Date",
    y="Buy_Hold_Value",
    title="Buy_Hold_Value Over Time",
    labels={"Date": "Date", "Buy_Hold_Value": "Buy_Hold_Value"},
    markers=True
)

fig.update_traces(line=dict(width=2))
fig.update_layout(width=1000,   # 🔑 width
                  height=500)    # 🔑 height

fig.show()

(2026-03-02 22:57:39,544) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:57:39,544) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:57:39,544) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:57:39,544) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:57:39,544) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:57:39,544) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
2026-03-02 22:57:39,544 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-03-02 22:57:39,983 - INFO - Fetched data for ABCAPITAL (2108 rows).
2026-03-02 22:57:40,465 - INFO - Fetched data for ANANDRATHI (1048 rows).
2026


Processing: 2023-04
Last Month Value: {}
            Date    Open    High     Low   Close   volume  oi     Ticker  \
1384  2023-04-03  154.00  154.90  152.00  153.80  2347046   0  ABCAPITAL   
1385  2023-04-05  153.80  156.45  152.80  155.25  3010848   0  ABCAPITAL   
1386  2023-04-06  155.70  159.30  153.80  157.90  3718974   0  ABCAPITAL   
1387  2023-04-10  158.75  160.30  155.90  159.75  3508046   0  ABCAPITAL   
1388  2023-04-11  160.50  161.30  157.05  157.60  2936298   0  ABCAPITAL   
...          ...     ...     ...     ...     ...      ...  ..        ...   
45351 2023-04-24  332.50  336.80  324.45  331.60   867129   0   TITAGARH   
45352 2023-04-25  330.00  352.90  329.05  336.85  4114109   0   TITAGARH   
45353 2023-04-26  339.00  342.70  329.15  332.35   709013   0   TITAGARH   
45354 2023-04-27  332.85  339.55  323.95  331.50   794444   0   TITAGARH   
45355 2023-04-28  333.00  335.20  329.25  332.25   351839   0   TITAGARH   

        %change  
1384   0.001628  
1385   0.

In [25]:
final_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046,0,ABCAPITAL,0.001628,3.756106,75.779854
1,2023-04-03,407.00,410.95,403.05,405.15,112564,0,ANANDRATHI,0.003219,3.762071,75.779854
2,2023-04-03,169.10,170.25,168.10,169.20,16659059,0,BANKBARODA,0.002073,3.757773,75.779854
3,2023-04-03,75.00,76.45,74.10,75.95,8102076,0,BANKINDIA,0.017415,3.815305,75.779854
4,2023-04-03,19500.00,19500.85,19200.00,19434.25,29923,0,BOSCHLTD,0.003322,3.762458,75.779854
...,...,...,...,...,...,...,...,...,...,...,...
15199,2026-03-02,158.99,168.21,158.80,165.59,21848406,0,SAIL,-0.000724,11.229498,214.282700
15200,2026-03-02,1185.00,1197.30,1179.40,1189.90,16270835,0,SBIN,-0.009819,12.726207,214.282700
15201,2026-03-02,1044.00,1071.00,1038.00,1052.50,6717872,0,SHRIRAMFIN,-0.024921,11.098493,214.282700
15202,2026-03-02,195.04,201.20,195.04,198.13,22256833,0,UNIONBANK,-0.020468,11.233752,214.282700


In [26]:
old_df = final_df[~((final_df['Date']>'2025-11-30') & (final_df['Ticker']=='GOLDBEES'))]
old_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046,0,ABCAPITAL,0.001628,3.756106,75.779854
1,2023-04-03,407.00,410.95,403.05,405.15,112564,0,ANANDRATHI,0.003219,3.762071,75.779854
2,2023-04-03,169.10,170.25,168.10,169.20,16659059,0,BANKBARODA,0.002073,3.757773,75.779854
3,2023-04-03,75.00,76.45,74.10,75.95,8102076,0,BANKINDIA,0.017415,3.815305,75.779854
4,2023-04-03,19500.00,19500.85,19200.00,19434.25,29923,0,BOSCHLTD,0.003322,3.762458,75.779854
...,...,...,...,...,...,...,...,...,...,...,...
15199,2026-03-02,158.99,168.21,158.80,165.59,21848406,0,SAIL,-0.000724,11.229498,214.282700
15200,2026-03-02,1185.00,1197.30,1179.40,1189.90,16270835,0,SBIN,-0.009819,12.726207,214.282700
15201,2026-03-02,1044.00,1071.00,1038.00,1052.50,6717872,0,SHRIRAMFIN,-0.024921,11.098493,214.282700
15202,2026-03-02,195.04,201.20,195.04,198.13,22256833,0,UNIONBANK,-0.020468,11.233752,214.282700


In [27]:
np.sort(old_df['Ticker'].unique())

array(['AAVAS', 'ABB', 'ABCAPITAL', 'ADANIPORTS', 'AIIL', 'AJANTPHARM',
       'AMBER', 'ANANDRATHI', 'ANANTRAJ', 'ANGELONE', 'APARINDS',
       'APLAPOLLO', 'ASHOKLEY', 'ASTERDM', 'AUBANK', 'AUROPHARMA',
       'AXISBANK', 'BAJAJ-AUTO', 'BAJAJHLDNG', 'BAJFINANCE', 'BANKBARODA',
       'BANKINDIA', 'BASF', 'BDL', 'BEL', 'BEML', 'BERGEPAINT',
       'BHARATFORG', 'BHARTIARTL', 'BHARTIHEXA', 'BHEL', 'BLUESTARCO',
       'BOSCHLTD', 'BPCL', 'BSE', 'BSOFT', 'CANBK', 'CHAMBLFERT',
       'CHENNPETRO', 'CHOLAFIN', 'CHOLAHLDNG', 'COCHINSHIP', 'COFORGE',
       'COHANCE', 'COLPAL', 'CONCORDBIO', 'COROMANDEL', 'CUB',
       'CUMMINSIND', 'CYIENT', 'DATAPATTNS', 'DBREALTY', 'DEEPAKFERT',
       'DELHIVERY', 'DIVISLAB', 'DIXON', 'DLF', 'DOMS', 'ECLERX',
       'EICHERMOT', 'EIHOTEL', 'ELECON', 'EMAMILTD', 'ENGINERSIN', 'ERIS',
       'ETERNAL', 'EXIDEIND', 'FEDERALBNK', 'FINCABLES', 'FLUOROCHEM',
       'FORTIS', 'GLAND', 'GLENMARK', 'GMDCLTD', 'GODFRYPHLP',
       'GODREJPROP', 'GOLDBEES', 'GPIL

In [28]:
df = fetch_truedata_history(
    ticker_list = ['GOLDBEES', 'SILVERBEES', 'MOGSEC'],
    duration = '5 Y',
    bar_size = 'EOD',
    sleep_time= 0.1
)[0]
df = df[["Date","Ticker", "Open", "Close"]]
df['%change'] = df.groupby('Ticker')['Close'].pct_change()
# January hedge logic (with SILVERBEES)
df_jan = df[(df['Date'] >= '2025-12-01') & (df['Date'] <= '2026-01-31')].copy()
# df.to_excel('C:\\Users\\Admin\\Momentum\\Automating Momentum True Data\\Trials\\nse200_Nifty_200_2025_Aug_nse200_nse200_nse200_nse200_nse200_returns.xlsx', index=False)
df_jan

(2026-03-02 22:59:49,536) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:49,536) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:49,536) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:49,536) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:49,536) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:49,536) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:49,536) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:49,536) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)


,Date,Ticker,Open,Close,%change
1177,2025-12-01,GOLDBEES,106.09,106.72,0.020073
1178,2025-12-02,GOLDBEES,106.65,105.63,-0.010214
1179,2025-12-03,GOLDBEES,106.25,106.51,0.008331
1180,2025-12-04,GOLDBEES,106.67,105.92,-0.005539
1181,2025-12-05,GOLDBEES,106.27,106.89,0.009158
...,...,...,...,...,...
3396,2026-01-23,MOGSEC,62.90,62.99,0.000953
3397,2026-01-27,MOGSEC,62.99,62.77,-0.003493
3398,2026-01-28,MOGSEC,62.94,62.99,0.003505
3399,2026-01-29,MOGSEC,62.83,62.90,-0.001429


In [29]:
# January hedge weights
ticker_weights_jan = {'GOLDBEES':0.6, 'SILVERBEES':0.2, 'MOGSEC':0.2}
factor = 51.2140199725867

ticker_value_jan = {ticker: weight * factor for ticker, weight in ticker_weights_jan.items()}
ticker_value_jan

{'GOLDBEES': 30.728411983552018,
 'SILVERBEES': 10.24280399451734,
 'MOGSEC': 10.24280399451734}

In [30]:
# January valuation (GOLDBEES + SILVERBEES + MOGSEC)
df_jan['BaseValue'] = df_jan['Ticker'].map(ticker_value_jan)
df_jan['Value'] = df_jan['Ticker'].map(ticker_value_jan)
df_jan['%change'] = pd.to_numeric(df_jan['%change'])
df_jan = df_jan.sort_values(['Ticker', 'Date'])
df_jan['ret_factor'] = 1 + df_jan['%change']
df_jan['cum_factor'] = df_jan.groupby('Ticker')['ret_factor'].cumprod()
df_jan['Value_On_Date'] = df_jan['BaseValue'] * df_jan['cum_factor']
df_jan = df_jan[['Date', 'Ticker', 'Open', 'Close', 'Value_On_Date', '%change']].rename(columns={'Value_On_Date':'Buy_Hold_Value'})

# Sell SILVERBEES in February; rebalance hedge to 40% GOLDBEES and 60% MOGSEC of the hedge book (25% of portfolio).
feb_factor = df_jan.groupby('Date', as_index=False)['Buy_Hold_Value'].sum().sort_values('Date')['Buy_Hold_Value'].iloc[-1]
df_feb = fetch_truedata_history(
    ticker_list = ['GOLDBEES', 'MOGSEC'],
    duration = '5 Y',
    bar_size = 'EOD',
    sleep_time= 0.1
)[0]
df_feb = fetch_truedata_history(
    ticker_list = ['GOLDBEES', 'MOGSEC'],
    duration = '5 Y',
    bar_size = 'EOD',
    sleep_time= 0.1
)[0]
df_feb = df_feb[["Date","Ticker", "Open", "Close"]]
df_feb = df_feb[(df_feb['Date'] >= '2026-02-01') & (df_feb['Date'] <= '2026-02-28')].copy()
df_feb['%change'] = df_feb.groupby('Ticker')['Close'].pct_change()
ticker_weights_feb = {'GOLDBEES':0.40, 'MOGSEC':0.60}
ticker_value_feb = {ticker: weight * feb_factor for ticker, weight in ticker_weights_feb.items()}
df_feb['BaseValue'] = df_feb['Ticker'].map(ticker_value_feb)
df_feb['Value'] = df_feb['Ticker'].map(ticker_value_feb)
df_feb['%change'] = pd.to_numeric(df_feb['%change'])
df_feb = df_feb.sort_values(['Ticker', 'Date'])
df_feb['ret_factor'] = 1 + df_feb['%change']
df_feb['cum_factor'] = df_feb.groupby('Ticker')['ret_factor'].cumprod()
df_feb['Value_On_Date'] = df_feb['BaseValue'] * df_feb['cum_factor']
df_feb = df_feb[['Date', 'Ticker', 'Open', 'Close', 'Value_On_Date', '%change']].rename(columns={'Value_On_Date':'Buy_Hold_Value'})

df = pd.concat([df_jan, df_feb], ignore_index=True).sort_values(['Date', 'Ticker'])
df

(2026-03-02 22:59:55,715) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:55,715) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:55,715) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:55,715) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:55,715) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:55,715) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:55,715) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 22:59:55,715) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)


,Date,Ticker,Open,Close,Buy_Hold_Value,%change
0,2025-12-01,GOLDBEES,106.09,106.72,31.345212,0.020073
42,2025-12-01,MOGSEC,62.70,62.82,10.220028,-0.002224
84,2025-12-01,SILVERBEES,166.02,166.21,10.859581,0.060216
1,2025-12-02,GOLDBEES,106.65,105.63,31.025064,-0.010214
43,2025-12-02,MOGSEC,62.97,62.90,10.233043,0.001273
...,...,...,...,...,...,...
165,2026-02-25,MOGSEC,64.02,63.53,39.699498,-0.007654
145,2026-02-26,GOLDBEES,132.69,130.48,28.798634,-0.009188
166,2026-02-26,MOGSEC,63.85,64.05,40.024443,0.008185
146,2026-02-27,GOLDBEES,130.70,131.60,29.045832,0.008584


In [31]:
conc_df = pd.concat([old_df, df])
conc_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046.0,0.0,ABCAPITAL,0.001628,3.756106,75.779854
1,2023-04-03,407.00,410.95,403.05,405.15,112564.0,0.0,ANANDRATHI,0.003219,3.762071,75.779854
2,2023-04-03,169.10,170.25,168.10,169.20,16659059.0,0.0,BANKBARODA,0.002073,3.757773,75.779854
3,2023-04-03,75.00,76.45,74.10,75.95,8102076.0,0.0,BANKINDIA,0.017415,3.815305,75.779854
4,2023-04-03,19500.00,19500.85,19200.00,19434.25,29923.0,0.0,BOSCHLTD,0.003322,3.762458,75.779854
...,...,...,...,...,...,...,...,...,...,...,...
165,2026-02-25,64.02,NaN,NaN,63.53,NaN,NaN,MOGSEC,-0.007654,39.699498,NaN
145,2026-02-26,132.69,NaN,NaN,130.48,NaN,NaN,GOLDBEES,-0.009188,28.798634,NaN
166,2026-02-26,63.85,NaN,NaN,64.05,NaN,NaN,MOGSEC,0.008185,40.024443,NaN
146,2026-02-27,130.70,NaN,NaN,131.60,NaN,NaN,GOLDBEES,0.008584,29.045832,NaN


In [32]:
import plotly.express as px

# ✅ Group by Date and calculate total portfolio value
portfolio_summary = (
    conc_df.groupby("Date", as_index=False)["Buy_Hold_Value"].sum()
)
# ✅ Plot with Plotly
fig = px.line(
    portfolio_summary,
    x="Date",
    y="Buy_Hold_Value",
    title="Buy_Hold_Value Over Time",
    labels={"Date": "Date", "Buy_Hold_Value": "Buy_Hold_Value"},
    markers=True
)

fig.update_traces(line=dict(width=2))
fig.update_layout(width=1000,   # 🔑 width
                  height=500)    # 🔑 height
fig.show()

In [33]:
# Momentum/Automating Momentum True Data/Trials/Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_returns.xlsx

In [34]:
conc_df.to_excel('C:\\Users\\anike\\Desktop\\Ocean_dev\\Momentum Handover\\Momentum Handover\\Trials\\Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_returns.xlsx', index=False)
# \Trials

In [35]:
nse = fetch_truedata_history(
    ticker_list = ['Nifty 500'],
    duration = '5 Y',
    bar_size = 'EOD',
    sleep_time= 0.1
)[0]
nse = nse[["Date", "Close"]].rename(columns={'Close':'Buy_Hold_Value'})
nse['%change'] = nse['Buy_Hold_Value'].pct_change()
nse = nse[nse['Date'] >= '2023-04-01']
nse.to_excel('Trials\\nse500_Nifty_500_2025_Apr_nse500_nse500_nse500_nse500_nse500_returns.xlsx', index=False)
nse

(2026-03-02 23:00:25,353) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 23:00:25,353) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 23:00:25,353) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 23:00:25,353) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 23:00:25,353) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 23:00:25,353) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 23:00:25,353) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)
(2026-03-02 23:00:25,353) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:23932 Thread:21948)


,Date,Buy_Hold_Value,%change
518,2023-04-03,14601.95,0.003029
519,2023-04-05,14709.40,0.007359
520,2023-04-06,14759.20,0.003386
521,2023-04-10,14790.55,0.002124
522,2023-04-11,14867.25,0.005186
...,...,...,...
1237,2026-02-24,23304.60,-0.007679
1238,2026-02-25,23403.80,0.004257
1239,2026-02-26,23448.50,0.001910
1240,2026-02-27,23166.85,-0.012011


In [36]:
conc_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046.0,0.0,ABCAPITAL,0.001628,3.756106,75.779854
1,2023-04-03,407.00,410.95,403.05,405.15,112564.0,0.0,ANANDRATHI,0.003219,3.762071,75.779854
2,2023-04-03,169.10,170.25,168.10,169.20,16659059.0,0.0,BANKBARODA,0.002073,3.757773,75.779854
3,2023-04-03,75.00,76.45,74.10,75.95,8102076.0,0.0,BANKINDIA,0.017415,3.815305,75.779854
4,2023-04-03,19500.00,19500.85,19200.00,19434.25,29923.0,0.0,BOSCHLTD,0.003322,3.762458,75.779854
...,...,...,...,...,...,...,...,...,...,...,...
165,2026-02-25,64.02,NaN,NaN,63.53,NaN,NaN,MOGSEC,-0.007654,39.699498,NaN
145,2026-02-26,132.69,NaN,NaN,130.48,NaN,NaN,GOLDBEES,-0.009188,28.798634,NaN
166,2026-02-26,63.85,NaN,NaN,64.05,NaN,NaN,MOGSEC,0.008185,40.024443,NaN
146,2026-02-27,130.70,NaN,NaN,131.60,NaN,NaN,GOLDBEES,0.008584,29.045832,NaN
